# GRPO with Kubeflow Trainer

This notebook demonstrates GRPO-style post-training for verifiable math reasoning on GSM8K using Kubeflow Trainer on Red Hat OpenShift AI.

It uses:
- `TRL GRPOTrainer`
- `TrainerClient` and `CustomTrainer`
- a Trainer-managed distributed runtime
- a deterministic reward based on the final numeric answer format (`#### <number>`)


## Install dependencies


In [ ]:
# !python3 -m pip install -U kubeflow trl datasets accelerate openai


In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import CustomTrainer, TrainerClient
from kubernetes import client as k8s

## Configure cluster access


In [ ]:
api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")
RUNTIME_NAME = "torch-distributed"

if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set.")
if NAMESPACE == "<your-namespace>":
    raise RuntimeError(
        "Set NOTEBOOK_NAMESPACE to your OpenShift AI project namespace before running this notebook."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)
backend_config = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_config)
runtime = client.backend.get_runtime(RUNTIME_NAME)
print(f"Using runtime: {runtime.name}")
print(f"Namespace: {NAMESPACE}")

## Define the GRPO training function


In [ ]:
def grpo_train():
    import json
    import os
    import re

    import torch
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer

    use_gpu = torch.cuda.is_available()
    rank = int(os.environ.get("RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if use_gpu:
        torch.cuda.set_device(local_rank)

    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    train_n = 128 if use_gpu else 16
    num_generations = 4 if use_gpu else 2

    def choose_microbatch(ws, use_cuda, generations):
        if not use_cuda:
            return 2, 1
        candidates = [(2, 2), (2, 1), (1, 4), (1, 2), (4, 1), (1, 1)]
        for per_device_bs, grad_accum in candidates:
            global_batch = per_device_bs * max(1, ws) * grad_accum
            if global_batch % generations == 0:
                return per_device_bs, grad_accum
        raise RuntimeError(
            f"No valid batch config for WORLD_SIZE={ws}, G={generations}"
        )

    per_device_bs, grad_accum = choose_microbatch(world_size, use_gpu, num_generations)
    dtype = (
        torch.bfloat16
        if use_gpu and torch.cuda.is_bf16_supported()
        else torch.float16
        if use_gpu
        else torch.float32
    )
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
    if use_gpu:
        model = model.to(torch.device(f"cuda:{local_rank}"))
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    def format_prompt(example):
        return {
            "prompt": [
                {
                    "role": "system",
                    "content": "Solve the math problem. End with #### followed by the numeric answer.",
                },
                {"role": "user", "content": example["question"]},
            ],
            "answer": example["answer"],
        }

    raw_train = load_dataset("openai/gsm8k", "main", split="train")
    raw_train = raw_train.select(range(min(train_n, len(raw_train))))
    train_ds = raw_train.map(format_prompt)

    def gsm8k_reward(completions, answer, **kwargs):
        rewards = []
        for completion, reference in zip(completions, answer, strict=False):
            content = (
                completion[0]["content"]
                if isinstance(completion, list)
                else str(completion)
            )
            reference_match = re.search(r"####\s*([\d,]+)", reference)
            reference_number = (
                reference_match.group(1).replace(",", "") if reference_match else None
            )
            prediction_match = re.search(r"####\s*([\d,]+)", content)
            prediction_number = (
                prediction_match.group(1).replace(",", "") if prediction_match else None
            )
            if (
                reference_number
                and prediction_number
                and reference_number == prediction_number
            ):
                rewards.append(1.0)
            elif prediction_match:
                rewards.append(0.1)
            else:
                rewards.append(0.0)
        return rewards

    config = GRPOConfig(
        output_dir="/tmp/grpo-gsm8k-output",
        num_train_epochs=1,
        per_device_train_batch_size=per_device_bs,
        gradient_accumulation_steps=grad_accum,
        num_generations=num_generations,
        max_prompt_length=256 if use_gpu else 128,
        max_completion_length=128 if use_gpu else 32,
        learning_rate=5e-6,
        beta=0.0,
        bf16=bool(use_gpu and torch.cuda.is_bf16_supported()),
        fp16=bool(use_gpu and not torch.cuda.is_bf16_supported()),
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        ddp_find_unused_parameters=False,
    )
    trainer = GRPOTrainer(
        model=model,
        args=config,
        reward_funcs=gsm8k_reward,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )
    train_output = trainer.train()
    if rank == 0:
        reward_logs = [
            entry for entry in trainer.state.log_history if "reward" in entry
        ]
        summary = {
            "model": model_name,
            "world_size": world_size,
            "cuda": use_gpu,
            "train_runtime_s": float(train_output.metrics.get("train_runtime", 0.0)),
            "train_samples": len(train_ds),
            "num_generations": num_generations,
        }
        if reward_logs:
            summary["first_reward_log"] = reward_logs[0]
            summary["last_reward_log"] = reward_logs[-1]
        print("[GRPO GSM8K] SPIKE_SUMMARY_JSON")
        print(json.dumps(summary, default=str))
        print("[GRPO GSM8K] Training complete")

## Submit the training job


In [ ]:
# Default: single node, single GPU
job_name = client.train(
    runtime=runtime,
    trainer=CustomTrainer(
        func=grpo_train,
        num_nodes=1,
        resources_per_node={"cpu": 4, "memory": "24Gi", "gpu": 1},
        packages_to_install=["trl", "datasets", "accelerate"],
    ),
)

# Single node, multi GPU
# job_name = client.train(runtime=runtime, trainer=CustomTrainer(func=grpo_train, num_nodes=1, resources_per_node={"cpu": 8, "memory": "48Gi", "gpu": 2}, packages_to_install=["trl", "datasets", "accelerate"]))

# Multi node, multi GPU
# job_name = client.train(runtime=runtime, trainer=CustomTrainer(func=grpo_train, num_nodes=2, resources_per_node={"cpu": 8, "memory": "48Gi", "gpu": 2}, packages_to_install=["trl", "datasets", "accelerate"]))

print(f"Job submitted: {job_name}")

In [ ]:
job = client.get_job(name=job_name)
print(f"Job: {job.name}")
print(f"Status: {job.status}")

In [ ]:
for line in client.get_job_logs(job_name, follow=True):
    print(line, end="")

## Cleanup

Delete the TrainJob when you are done validating the run.


In [ ]:
# client.delete_job(name=job_name)
